# Tutorial 4: Cross-Slice Evaluation

This notebook demonstrates CauST's **multi-slice** mode and the **Leave-One-Donor-Out (LODO)** validation protocol — the key benchmark for measuring cross-slice generalization.

## Why Cross-Slice?

A model that works on Slice 1 but fails on Slice 2 (from a different donor) is not reliable. CauST's invariance scoring specifically addresses this by finding genes that are *consistently causal* across all slices and donors.

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import anndata as ad
from scipy.sparse import csr_matrix
import matplotlib.pyplot as plt
import pandas as pd

from caust import CauST
from caust.data.loader import load_and_preprocess
from caust.evaluate.metrics import evaluate_single_slice, compute_ari
from caust.causal.invariance import lodo_splits

/workspace/CauST/caust_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Create Multi-Donor Synthetic Data

We simulate the DLPFC-like setup: 3 donors, 2 slices each, 3 spatial domains.
Each donor has slightly different expression profiles (biological variability).

In [2]:
def make_slice(n_obs=200, n_vars=150, n_domains=3, donor_shift=0.0, seed=42):
    """Create a single tissue slice with donor-specific shifts."""
    rng = np.random.default_rng(seed)
    obs_per = n_obs // n_domains

    X_parts, coord_parts = [], []
    for i in range(n_domains):
        base = rng.poisson(lam=5 + donor_shift, size=(obs_per, n_vars)).astype(np.float32)
        # Causal genes (consistent across donors)
        causal_start = i * 10
        base[:, causal_start:causal_start + 10] += 10 * (i + 1)
        # Confounding genes (donor-specific — NOT causal for domains)
        conf_start = 100 + i * 5
        base[:, conf_start:conf_start + 5] += donor_shift * 3
        X_parts.append(base)
        angle = 2 * np.pi * i / n_domains
        coord_parts.append(rng.normal([12 * np.cos(angle), 12 * np.sin(angle)], 2, (obs_per, 2)))

    X = np.vstack(X_parts)
    adata = ad.AnnData(X=csr_matrix(X))
    adata.obs_names = [f'spot_{i}' for i in range(X.shape[0])]
    adata.var_names = [f'gene_{j}' for j in range(n_vars)]
    adata.obsm['spatial'] = np.vstack(coord_parts)
    adata.obs['ground_truth'] = np.repeat(np.arange(n_domains), obs_per).astype(str)
    return adata

# 3 donors × 2 slices each = 6 slices
slices = {}
donor_map = {}
seed_counter = 0
for donor_idx, donor_name in enumerate(['Donor1', 'Donor2', 'Donor3']):
    donor_shift = donor_idx * 2.0  # each donor has different baseline expression
    for s in range(2):
        sid = f'{donor_name}_s{s+1}'
        slices[sid] = make_slice(donor_shift=donor_shift, seed=seed_counter)
        donor_map[sid] = donor_name
        seed_counter += 10

print(f'Created {len(slices)} slices from {len(set(donor_map.values()))} donors:')
for sid, adata in slices.items():
    print(f'  {sid} (donor={donor_map[sid]}): {adata.n_obs} spots × {adata.n_vars} genes')

Created 6 slices from 3 donors:
  Donor1_s1 (donor=Donor1): 198 spots × 150 genes
  Donor1_s2 (donor=Donor1): 198 spots × 150 genes
  Donor2_s1 (donor=Donor2): 198 spots × 150 genes
  Donor2_s2 (donor=Donor2): 198 spots × 150 genes
  Donor3_s1 (donor=Donor3): 198 spots × 150 genes
  Donor3_s2 (donor=Donor3): 198 spots × 150 genes


## 2. Preprocess All Slices

In [3]:
slices_pp = {}
for sid, adata in slices.items():
    slices_pp[sid] = load_and_preprocess(adata, n_top_genes=100, min_genes=5, min_cells=1)
    print(f'  {sid}: {slices_pp[sid].n_obs} spots × {slices_pp[sid].n_vars} genes')

[loader] Received AnnData directly
         Raw shape: 198 spots × 150 genes
         After QC : 198 spots × 150 genes


         HVGs selected: 100
         Final shape : 198 spots × 100 genes

  Donor1_s1: 198 spots × 100 genes
[loader] Received AnnData directly
         Raw shape: 198 spots × 150 genes
         After QC : 198 spots × 150 genes
         HVGs selected: 100
         Final shape : 198 spots × 100 genes

  Donor1_s2: 198 spots × 100 genes
[loader] Received AnnData directly
         Raw shape: 198 spots × 150 genes
         After QC : 198 spots × 150 genes
         HVGs selected: 100
         Final shape : 198 spots × 100 genes

  Donor2_s1: 198 spots × 100 genes
[loader] Received AnnData directly
         Raw shape: 198 spots × 150 genes
         After QC : 198 spots × 150 genes
         HVGs selected: 100
         Final shape : 198 spots × 100 genes

  Donor2_s2: 198 spots × 100 genes
[loader] Received AnnData directly
         Raw shape: 198 spots × 150 genes
         After QC : 198 spots × 150 genes
         HVGs selected: 100
         Final shape : 198 spots × 100 genes

  Donor3_s1: 1

## 3. Run CauST Multi-Slice Mode

This is the **recommended** CauST usage. It:
1. Trains a model per slice
2. Computes per-slice causal scores
3. Computes IRM-style invariance scores across all slices
4. Combines causal + invariance → final gene ranking

In [4]:
model = CauST(
    n_causal_genes=40, n_clusters=3, epochs=60,
    scoring_method='gradient', alpha=0.5,
    filter_mode='filter_and_reweight', verbose=True,
)

results = model.fit_multi_slice(slices_pp, donor_map=donor_map)

print(f'\nResults for {len(results)} slices.')


  CauST initialized
  device         : cuda
  n_causal_genes : 40
  filter_mode    : filter_and_reweight
  scoring_method : gradient
  intervention   : mean_impute
  alpha          : 0.5

Multi-slice fit: 6 slices

  ── Slice 1/6: Donor1_s1 ──
[graph] Building KNN graph: 198 spots, k=6 neighbours
[graph] Edges: 1,520  (avg degree 7.7)



    Training:   0%|                                                                                                     | 0/60 [00:00<?]

    Training:   0%|                                                                                                     | 0/60 [00:00<?]

    Training:   2%|█▌                                                                                               | 1/60 [00:00<00:13]

    Training:   2%|█▌                                                                                               | 1/60 [00:00<00:13]

...

    Training: 100%|████████████████████████████████████████████████████████████████████████████████████████████████| 60/60 [00:00<00:00]

  Training complete. Final loss: 0.686916
[scorer] Gradient scoring done. Top-5: gene_42(1.000), gene_115(0.961), gene_143(0.905), gene_32(0.719), gene_124(0.715)

Computing invariance scores across all slices …
[invariance] Scored 150 genes across 6 slices.
  Top-5 invariant genes: gene_99(1.000), gene_31(0.782), gene_115(0.767), gene_137(0.740), gene_35(0.733)
[invariance] Final combined scores (alpha=0.5).
  Top-5: gene_144(1.000), gene_35(0.995), gene_61(0.995), gene_31(0.973), gene_93(0.931)
[invariance] Cross-donor correlation  (3 pairs):  Pearson r = 0.155,  Spearman ρ = 0.019
Multi-slice fitting complete.
[filter] Hard filter: 40 / 100 genes kept  (requested k=40)
[filter] Soft reweight: 40 genes reweighted (0 genes with zero score effectively silenced)
[filter] CauST filter+reweight: 40 genes, expression reweighted by causal score.


[filter] Hard filter: 40 / 100 genes kept  (requested k=40)
[filter] Soft reweight: 40 genes reweighted (0 genes with zero score effectively silenced)
[filter] CauST filter+reweight: 40 genes, expression reweighted by causal score.
[filter] Hard filter: 40 / 100 genes kept  (requested k=40)
[filter] Soft reweight: 40 genes reweighted (0 genes with zero score effectively silenced)
[filter] CauST filter+reweight: 40 genes, expression reweighted by causal score.
[filter] Hard filter: 40 / 100 genes kept  (requested k=40)
[filter] Soft reweight: 40 genes reweighted (0 genes with zero score effectively silenced)
[filter] CauST filter+reweight: 40 genes, expression reweighted by causal score.
[filter] Hard filter: 40 / 100 genes kept  (requested k=40)
[filter] Soft reweight: 40 genes reweighted (0 genes with zero score effectively silenced)
[filter] CauST filter+reweight: 40 genes, expression reweighted by causal score.


[filter] Hard filter: 40 / 100 genes kept  (requested k=40)
[filter] Soft reweight: 40 genes reweighted (0 genes with zero score effectively silenced)
[filter] CauST filter+reweight: 40 genes, expression reweighted by causal score.



Results for 6 slices.


## 4. Evaluate Per-Slice Performance

In [5]:
rows = []
for sid, adata_out in results.items():
    labels_pred = adata_out.obs['caust_domain'].astype(int).values
    latent_Z = adata_out.obsm['caust_latent']
    labels_true = slices_pp[sid].obs.loc[adata_out.obs_names, 'ground_truth'].values
    m = evaluate_single_slice(labels_pred, latent_Z, labels_true)
    m['slice'] = sid
    m['donor'] = donor_map[sid]
    rows.append(m)

df = pd.DataFrame(rows)
print(df.to_string(index=False))

 silhouette  ari  nmi     slice  donor
   0.483996  1.0  1.0 Donor1_s1 Donor1
   0.483879  1.0  1.0 Donor1_s2 Donor1
   0.552701  1.0  1.0 Donor2_s1 Donor2
   0.601643  1.0  1.0 Donor2_s2 Donor2
   0.629019  1.0  1.0 Donor3_s1 Donor3
   0.715458  1.0  1.0 Donor3_s2 Donor3


## 5. Leave-One-Donor-Out (LODO) Validation

The gold-standard cross-generalization test:
- Train CauST on all slices from 2 donors
- Test on slices from the held-out donor
- Rotate which donor is held out

In [6]:
slice_ids = list(slices_pp.keys())
splits = lodo_splits(slice_ids, donor_map)

lodo_rows = []
for train_sids, test_sids in splits:
    test_donor = donor_map[test_sids[0]]
    print(f'\n--- LODO: hold out {test_donor} ---')
    print(f'    Train: {train_sids}')
    print(f'    Test:  {test_sids}')

    # Train on train slices
    train_slices = {sid: slices_pp[sid].copy() for sid in train_sids}
    lodo_model = CauST(
        n_causal_genes=40, n_clusters=3, epochs=60,
        scoring_method='gradient', alpha=0.5,
        filter_mode='filter_and_reweight', verbose=False,
    )
    lodo_model.fit_multi_slice(train_slices)

    # Test on held-out donor's slices
    for sid in test_sids:
        adata_test = lodo_model.transform(slices_pp[sid].copy())
        labels_pred = adata_test.obs['caust_domain'].astype(int).values
        latent_Z = adata_test.obsm['caust_latent']
        labels_true = slices_pp[sid].obs.loc[adata_test.obs_names, 'ground_truth'].values
        m = evaluate_single_slice(labels_pred, latent_Z, labels_true)
        m['test_slice'] = sid
        m['held_out_donor'] = test_donor
        lodo_rows.append(m)
        print(f'    {sid}: ARI={m.get("ari", "N/A"):.4f}')

lodo_df = pd.DataFrame(lodo_rows)
print('\n=== LODO Results ===')
print(lodo_df.to_string(index=False))

  LODO split: test_donor=Donor1  train=['Donor2_s1', 'Donor2_s2', 'Donor3_s1', 'Donor3_s2']  test=['Donor1_s1', 'Donor1_s2']
  LODO split: test_donor=Donor2  train=['Donor1_s1', 'Donor1_s2', 'Donor3_s1', 'Donor3_s2']  test=['Donor2_s1', 'Donor2_s2']
  LODO split: test_donor=Donor3  train=['Donor1_s1', 'Donor1_s2', 'Donor2_s1', 'Donor2_s2']  test=['Donor3_s1', 'Donor3_s2']

--- LODO: hold out Donor1 ---
    Train: ['Donor2_s1', 'Donor2_s2', 'Donor3_s1', 'Donor3_s2']
    Test:  ['Donor1_s1', 'Donor1_s2']


  Training complete. Final loss: 0.738008
[scorer] Gradient scoring done. Top-5: gene_148(1.000), gene_30(0.986), gene_48(0.963), gene_34(0.932), gene_63(0.917)


  Training complete. Final loss: 0.728189
[scorer] Gradient scoring done. Top-5: gene_82(1.000), gene_7(0.946), gene_124(0.941), gene_80(0.887), gene_95(0.872)


  Training complete. Final loss: 0.709058
[scorer] Gradient scoring done. Top-5: gene_91(1.000), gene_116(0.808), gene_88(0.789), gene_148(0.766), gene_44(0.761)


  Training complete. Final loss: 0.689653
[scorer] Gradient scoring done. Top-5: gene_115(1.000), gene_78(0.941), gene_35(0.936), gene_69(0.917), gene_133(0.878)
[invariance] Scored 148 genes across 4 slices.
  Top-5 invariant genes: gene_115(1.000), gene_34(0.932), gene_55(0.856), gene_136(0.802), gene_97(0.781)
[invariance] Final combined scores (alpha=0.5).
  Top-5: gene_148(1.000), gene_124(0.928), gene_48(0.906), gene_91(0.893), gene_90(0.882)
[filter] Hard filter: 40 / 100 genes kept  (requested k=40)
[filter] Soft reweight: 40 genes reweighted (0 genes with zero score effectively silenced)
[filter] CauST filter+reweight: 40 genes, expression reweighted by causal score.
[filter] Hard filter: 40 / 100 genes kept  (requested k=40)
[filter] Soft reweight: 40 genes reweighted (0 genes with zero score effectively silenced)
[filter] CauST filter+reweight: 40 genes, expression reweighted by causal score.


...

  Training complete. Final loss: 0.729872
[scorer] Gradient scoring done. Top-5: gene_138(1.000), gene_55(0.917), gene_35(0.851), gene_70(0.849), gene_96(0.839)


  Training complete. Final loss: 0.726935
[scorer] Gradient scoring done. Top-5: gene_32(1.000), gene_147(0.928), gene_39(0.885), gene_74(0.723), gene_139(0.661)
[invariance] Scored 148 genes across 4 slices.
  Top-5 invariant genes: gene_138(1.000), gene_96(0.983), gene_32(0.965), gene_51(0.937), gene_39(0.847)
[invariance] Final combined scores (alpha=0.5).
  Top-5: gene_138(1.000), gene_134(0.932), gene_147(0.914), gene_82(0.854), gene_96(0.845)
[filter] Hard filter: 40 / 100 genes kept  (requested k=40)
[filter] Soft reweight: 40 genes reweighted (0 genes with zero score effectively silenced)
[filter] CauST filter+reweight: 40 genes, expression reweighted by causal score.


[filter] Hard filter: 40 / 100 genes kept  (requested k=40)
[filter] Soft reweight: 40 genes reweighted (0 genes with zero score effectively silenced)
[filter] CauST filter+reweight: 40 genes, expression reweighted by causal score.
[filter] Hard filter: 40 / 100 genes kept  (requested k=40)
[filter] Soft reweight: 40 genes reweighted (0 genes with zero score effectively silenced)
[filter] CauST filter+reweight: 40 genes, expression reweighted by causal score.


[filter] Hard filter: 40 / 100 genes kept  (requested k=40)
[filter] Soft reweight: 40 genes reweighted (0 genes with zero score effectively silenced)
[filter] CauST filter+reweight: 40 genes, expression reweighted by causal score.
[filter] Hard filter: 40 / 100 genes kept  (requested k=40)
[filter] Soft reweight: 40 genes reweighted (0 genes with zero score effectively silenced)
[filter] CauST filter+reweight: 40 genes, expression reweighted by causal score.
    Donor3_s1: ARI=1.0000


[filter] Hard filter: 40 / 100 genes kept  (requested k=40)
[filter] Soft reweight: 40 genes reweighted (0 genes with zero score effectively silenced)
[filter] CauST filter+reweight: 40 genes, expression reweighted by causal score.
    Donor3_s2: ARI=1.0000

=== LODO Results ===
 silhouette  ari  nmi test_slice held_out_donor
   0.492967  1.0  1.0  Donor1_s1         Donor1
   0.573333  1.0  1.0  Donor1_s2         Donor1
   0.516381  1.0  1.0  Donor2_s1         Donor2
   0.517215  1.0  1.0  Donor2_s2         Donor2
   0.634572  1.0  1.0  Donor3_s1         Donor3
   0.639964  1.0  1.0  Donor3_s2         Donor3


## 6. Inspect Invariant Causal Genes

Genes ranked high by CauST should be the **truly causal** ones (gene_0–gene_29) that are consistent across donors, NOT the confounding genes (gene_100+) that vary by donor.

In [7]:
top = model.get_top_causal_genes(n=20)
print('Top 20 genes after invariance scoring:')
for gene, score in top:
    marker = ''
    idx = int(gene.split('_')[1])
    if idx < 30:
        marker = '  ← CAUSAL (domain marker)'
    elif idx >= 100:
        marker = '  ← confounding (donor-specific)'
    print(f'  {gene}: {score:.4f}{marker}')

Top 20 genes after invariance scoring:
  gene_144: 1.0000  ← confounding (donor-specific)
  gene_35: 0.9952
  gene_61: 0.9947
  gene_31: 0.9731
  gene_93: 0.9308
  gene_137: 0.9193  ← confounding (donor-specific)
  gene_44: 0.9153
  gene_42: 0.8979
  gene_148: 0.8776  ← confounding (donor-specific)
  gene_99: 0.8756
  gene_82: 0.8665
  gene_72: 0.8664
  gene_141: 0.8642  ← confounding (donor-specific)
  gene_92: 0.8592
  gene_143: 0.8438  ← confounding (donor-specific)
  gene_73: 0.8338
  gene_103: 0.8334  ← confounding (donor-specific)
  gene_58: 0.8269
  gene_126: 0.8264  ← confounding (donor-specific)
  gene_33: 0.8218


## 7. Visualize Cross-Slice Heatmap

In [8]:
import os
os.makedirs('output', exist_ok=True)

if hasattr(model, 'per_slice_scores'):
    from caust.visualize.plots import plot_invariance_heatmap
    plot_invariance_heatmap(
        model.per_slice_scores,
        top_k=30,
        title='Gene Causal Scores Across 6 Slices',
        out_path='output/cross_slice_heatmap.png',
    )
    print('Heatmap saved to output/cross_slice_heatmap.png')
else:
    print('No per_slice_scores found — run fit_multi_slice first.')

[plots] Saved: output/cross_slice_heatmap.png
Heatmap saved to output/cross_slice_heatmap.png


## 8. Visualize LODO ARI Comparison

In [9]:
if 'ari' in lodo_df.columns:
    fig, ax = plt.subplots(figsize=(8, 4))
    donors = lodo_df['held_out_donor'].unique()
    x = np.arange(len(donors))
    means = [lodo_df[lodo_df['held_out_donor'] == d]['ari'].mean() for d in donors]
    bars = ax.bar(x, means, color='steelblue', alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(donors)
    ax.set_ylabel('ARI (held-out donor)')
    ax.set_title('LODO Cross-Donor Generalization')
    ax.set_ylim(0, 1.05)
    for bar, m in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2, m + 0.02, f'{m:.3f}', ha='center', fontsize=10)
    plt.tight_layout()
    plt.show()

## Summary

| Protocol | What it tests | Why it matters |
|----------|--------------|----------------|
| Same-slice ARI | Quality on training data | Basic sanity check |
| **LODO ARI** | Generalization to unseen donors | **CauST's main value proposition** |
| Invariance heatmap | Visual confirmation of gene stability | Biological interpretability |

CauST's invariance scoring specifically penalises donor-specific confounders and promotes universally causal genes, directly improving LODO performance.